# Decoding + Tangling v2 — Corrected Analysis

Fixes vs `07_decoding.ipynb`:
1. **Noise at eval** — `noise_at_eval=True` to break ceiling decoding effect
2. **Corrected pre-target window** — `(-300, -100)` ms instead of `(-300, 0)`
3. **Loads v2 checkpoint** (if available, falls back to v1)
4. **Both pre-target windows** compared side by side

What to look for:
- Pre-target: whether a quadratic fit beats linear
- Post-target: whether a quadratic fit beats linear
- Whether tangling tracks decodable position info (Spearman correlation)
- Whether post-target tangling vs CTOA is monotone (linear fit)

In [ ]:
import sys, os, pathlib
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

from src.model import BioLeakyRNN
from src.env import CuedTargetWithDistractorsV3
from src.analysis import (
    collect_trials,
    decode_position_by_ctoa_bin,
    tangling_by_ctoa_bin,
    polynomial_regression,
)
from src.plotting import (
    plot_decoding_vs_ctoa,
    plot_decoding_timecourse,
    plot_tangling_timecourses,
    plot_tangling_vs_ctoa,
    plot_tangling_vs_position_info,
    plot_tangling_vs_rt,
)

device = "cpu"
print("device:", device)

## 1. Load model

In [ ]:
ckpt_v2 = Path("checkpoints/stage2_v2.pt")
ckpt_v1 = Path("checkpoints/stage2.pt")

if ckpt_v2.exists():
    ckpt_path = ckpt_v2
    sigma_rec = 0.10
    print("Using v2 checkpoint")
else:
    ckpt_path = ckpt_v1
    sigma_rec = 0.05
    print("v2 checkpoint not found, using v1")


def make_model():
    return BioLeakyRNN(
        input_size=7,
        hidden_size=128,
        output_size=2,
        dt=20.0,
        tau=100.0,
        activation="softplus",
        sigma_rec=sigma_rec,
        use_ei=True,
        exc_ratio=0.7,
        use_dale=True,
        mask_seed=42,
    )


def make_env():
    return CuedTargetWithDistractorsV3(
        dt=20, cue_strength=1.0, p_distractor_trial=0.6, distractor_strength=1.0
    )


model = make_model().to(device)
model.load_state_dict(torch.load(str(ckpt_path), weights_only=True)["state_dict"])
model.eval()
model.noise_at_eval = True
print(f"Loaded {ckpt_path.name}, noise_at_eval={model.noise_at_eval}")

## 2. Collect trials (with noise)

In [ ]:
print("Collecting 5000 trials (with recurrent noise)...")
trials = collect_trials(model, make_env, n_trials=5000, device=device)

outcomes = {}
for tr in trials:
    o = tr["train_outcome"]
    outcomes[o] = outcomes.get(o, 0) + 1
total = len(trials)
print(f"Total: {total}")
for o, n in sorted(outcomes.items(), key=lambda x: -x[1]):
    print(f"  {o}: {n} ({100*n/total:.1f}%)")

correct_trials = [tr for tr in trials if tr["train_outcome"] == "correct"]
print(f"\nCorrect trials: {len(correct_trials)}")

## 3. PCA dims sweep (with noise — expect lower ceiling)

With noise at eval, decoding should no longer be at 1.0 everywhere.

In [ ]:
dims_to_try = [None, 32, 16, 10, 6, 4, 2]

# Pre-target window: (-300, -100) ms
print("--- Pre-target (-300..-100ms) decoding ---")
print(f'{"K":>5}  ' + "  ".join(f"b{i}" for i in range(10)))
for k in dims_to_try:
    res = decode_position_by_ctoa_bin(
        correct_trials,
        window_ms=(-300, -100),
        align_key="target_onset",
        outcome=None,
        min_trials=10,
        n_splits=5,
        pca_dims=k,
    )
    accs = "  ".join(f"{a:.2f}" for a in res["acc_per_bin"])
    label = str(k) if k is not None else "all"
    print(f"{label:>5}  {accs}")

print()
print("--- Post-target (100..300ms) decoding ---")
print(f'{"K":>5}  ' + "  ".join(f"b{i}" for i in range(10)))
for k in dims_to_try:
    res = decode_position_by_ctoa_bin(
        correct_trials,
        window_ms=(100, 300),
        align_key="target_onset",
        outcome=None,
        min_trials=10,
        n_splits=5,
        pca_dims=k,
    )
    accs = "  ".join(f"{a:.2f}" for a in res["acc_per_bin"])
    label = str(k) if k is not None else "all"
    print(f"{label:>5}  {accs}")

## 4. Main decoding analysis

Choose pca_dims based on the sweep above — pick the K where CTOA modulation is visible.

In [ ]:
PCA_DIMS = 6  # adjust based on sweep

# Pre-target window: (-300, -100) ms
dec_pre = decode_position_by_ctoa_bin(
    correct_trials,
    window_ms=(-300, -100),
    align_key="target_onset",
    outcome=None,
    min_trials=10,
    n_splits=5,
    pca_dims=PCA_DIMS,
)

dec_post = decode_position_by_ctoa_bin(
    correct_trials,
    window_ms=(100, 300),
    align_key="target_onset",
    outcome=None,
    min_trials=10,
    n_splits=5,
    pca_dims=PCA_DIMS,
)

print(f"Decoding with pca_dims={PCA_DIMS}, noise_at_eval=True")
print(f"Pre-target window: (-300, -100) ms")
print()
print("Pre-target decoding per CTOA bin:")
for b, acc, n in zip(dec_pre["labels"], dec_pre["acc_per_bin"], dec_pre["n_per_bin"]):
    print(f"  bin {b:2d}: acc={acc:.3f}  (n={n})")
print()
print("Post-target decoding per CTOA bin:")
for b, acc, n in zip(dec_post["labels"], dec_post["acc_per_bin"], dec_post["n_per_bin"]):
    print(f"  bin {b:2d}: acc={acc:.3f}  (n={n})")

## 5. Regression: decoding accuracy vs CTOA

In [ ]:
plot_decoding_vs_ctoa(dec_pre, dec_post)

In [ ]:
print("=== Regression analysis ===")
for label, dec in [
    ("Pre-target (-300..-100)", dec_pre),
    ("Post-target (100..300)", dec_post),
]:
    x = dec["ctoa_ms_mean"]
    y = dec["acc_per_bin"]
    n = len(x)
    print(f"\n{label}:")
    for deg in [1, 2]:
        reg = polynomial_regression(x, y, degree=deg)
        if reg["y_hat"] is not None:
            ss_res = np.sum((reg["y"] - reg["y_hat"]) ** 2)
            k = deg + 1
            aic = n * np.log(ss_res / n + 1e-30) + 2 * k
            print(
                f'  deg-{deg}: R2={reg["r2"]:.3f}  p={reg["p_value"]:.4f}  AIC={aic:.2f}'
            )
        else:
            print(f"  deg-{deg}: insufficient data")

pass

## 6. Decoding timecourse

In [ ]:
plot_decoding_timecourse(
    correct_trials,
    align_key="target_onset",
    window_step_ms=40,
    window_width_ms=100,
    ctoa_bin_groups={
        "early CTOA (bins 0-2)": list(range(0, 3)),
        "mid   CTOA (bins 4-6)": list(range(4, 7)),
        "late  CTOA (bins 7-9)": list(range(7, 10)),
    },
    outcome=None,
    pca_dims=PCA_DIMS,
)

## 7. Tangling analysis

In [ ]:
tang_pre = tangling_by_ctoa_bin(
    correct_trials,
    align_key="target_onset",
    window_before=15,  # 300 ms
    window_after=0,
    pca_dims=20,
    outcome=None,
    min_trials=10,
)

tang_post = tangling_by_ctoa_bin(
    correct_trials,
    align_key="target_onset",
    window_before=0,
    window_after=15,  # 300 ms
    pca_dims=20,
    outcome=None,
    min_trials=10,
)

if tang_pre is not None:
    print("Pre-target tangling per CTOA bin:")
    for b, q in zip(tang_pre["labels"], tang_pre["Q_mean"]):
        print(f"  bin {b}: Q={q:.3f}")

if tang_post is not None:
    print("\nPost-target tangling per CTOA bin:")
    for b, q in zip(tang_post["labels"], tang_post["Q_mean"]):
        print(f"  bin {b}: Q={q:.3f}")

In [ ]:
if tang_pre is not None and tang_post is not None:
    plot_tangling_timecourses(tang_pre, tang_post)
    plot_tangling_vs_ctoa(tang_pre, tang_post)

## 8. Tangling vs position info (Spearman correlation)

In [ ]:
if tang_pre is not None:
    common_bins = sorted(set(dec_pre["labels"]) & set(tang_pre["labels"]))
    dec_idx = [dec_pre["labels"].index(b) for b in common_bins]
    tang_idx = [tang_pre["labels"].index(b) for b in common_bins]

    dec_acc_aligned = dec_pre["acc_per_bin"][dec_idx]
    tang_Q_aligned = tang_pre["Q_mean"][tang_idx]

    print(f"Common bins: {common_bins}")
    print(f"Decoding acc: {np.round(dec_acc_aligned, 3)}")
    print(f"Tangling Q:   {np.round(tang_Q_aligned, 3)}")

    tang_pre_aligned = {"Q_mean": tang_Q_aligned, "labels": common_bins}
    fig, rho, p = plot_tangling_vs_position_info(tang_pre_aligned, dec_acc_aligned)
    print(f"\nSpearman rho = {rho:.3f}, p = {p:.4f}")

## 9. Tangling vs reaction time

In [ ]:
from collections import defaultdict

rt_by_bin = defaultdict(list)
for tr in correct_trials:
    b = tr.get("ctoa_bin")
    rt = tr.get("rt_ms")
    if b is not None and rt is not None:
        rt_by_bin[b].append(rt)

rt_mean = {b: np.mean(rts) for b, rts in rt_by_bin.items()}
print("Mean RT per CTOA bin:")
for b in sorted(rt_mean):
    print(f"  bin {b}: RT={rt_mean[b]:.1f} ms  (n={len(rt_by_bin[b])})")

if tang_post is not None:
    plot_tangling_vs_rt(tang_post, rt_mean)

## 10. Comparison: old window (-300,0) vs corrected window (-300,-100)

In [ ]:
dec_pre_old = decode_position_by_ctoa_bin(
    correct_trials,
    window_ms=(-300, 0),  # old window
    align_key="target_onset",
    outcome=None,
    min_trials=10,
    n_splits=5,
    pca_dims=PCA_DIMS,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, dec, title in [
    (axes[0], dec_pre, "Pre-target (-300, -100) ms [CORRECTED]"),
    (axes[1], dec_pre_old, "Pre-target (-300, 0) ms [OLD]"),
]:
    x = dec["ctoa_ms_mean"]
    y = dec["acc_per_bin"]
    ax.errorbar(x, y, yerr=dec["sem_per_bin"], fmt="o-", capsize=3)
    ax.axhline(0.25, ls="--", color="gray", alpha=0.5, label="chance")

    for deg, c in [(1, "blue"), (2, "red")]:
        reg = polynomial_regression(x, y, degree=deg)
        if reg["y_hat"] is not None:
            order = np.argsort(reg["x"])
            ax.plot(
                reg["x"][order],
                reg["y_hat"][order],
                "--",
                color=c,
                label=f'deg-{deg}: R2={reg["r2"]:.2f} p={reg["p_value"]:.3f}',
            )

    ax.set_xlabel("CTOA (ms)")
    ax.set_ylabel("Decoding accuracy")
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle(f"Window comparison (pca_dims={PCA_DIMS}, noise_at_eval)", fontsize=13)
plt.tight_layout()
plt.show()

## Summary

| Metric | v1 (no noise) | v2 (noise at eval) |
|--------|---------------|--------------------|
| Pre-target linear R2 | ~0.5 (ceiling) | ? |
| Pre-target quadratic R2 | ~0.7 (ceiling) | ? |
| Post-target linear R2 | ~0.2 (ceiling) | ? |
| Post-target quadratic R2 | ~0.2 (ceiling) | ? |
| Tangling-decoding rho | low (ceiling) | ? |
| Tangling-RT rho | ? | ? |